# Credit Cluster Scoring Tool

Use this notebook to score a private, simulated, or competitor company against the saved credit clustering model.

Expected workflow:

1. Put the saved artifact at `saved_models/credit_cluster_model_v1.joblib`.
2. Enter company financials manually, or load a CSV/Excel file.
3. Run the scoring cells.
4. Review cluster assignment, near-default affinity, feature coverage, and warning flags.

Important: the output is **not a formal credit rating** and not a calibrated probability of default. It is a public-company financial profile match based on distance to the learned KMeans credit clusters.

In [1]:
#Import your libraries here

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import sys

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Load saved fitted model artifact

The training notebook should have saved this file after fitting the KMeans model.

In [2]:
# Import project modules and load trained artifact

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils import (
    build_credit_report_tables,
    save_credit_report_outputs,
    save_credit_pdf_report,
)

from src.credit_clustering import (
    # Artifact utilities
    load_artifact,
    get_segment_artifact,
    infer_near_default_cluster_from_artifact,

    # Scoring and diagnostics
    score_companies,
    add_adjacent_bucket_distances_and_outlook,
    compare_to_cluster_profile,
    make_scenarios,

    #Analyst guardrails
    apply_credit_guardrails,

    # Config
    DEFAULT_PRIMARY_SEGMENT,
    DEFAULT_SCORING_TEMPERATURE,
    DEFAULT_FX_TO_MODEL_CURRENCY,
    DEFAULT_SCORING_MIN_DENOMINATOR,
    RATIO_COLS,
    SUMMARY_COLS_WITH_OUTLOOK,
    SCENARIO_SUMMARY_COLS,
)

MODEL_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "saved_models"
    / "nonfinancial_credit_scorecard_kmeans_k5_v3.joblib"
)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model artifact not found at {MODEL_PATH}")

INPUT_PATH = PROJECT_ROOT / "inputs"
INPUT_PATH.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROJECT_ROOT / "outputs" / "company_scores"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

artifact = load_artifact(MODEL_PATH)

SCORING_SEGMENT = artifact.get("primary_segment", DEFAULT_PRIMARY_SEGMENT)
SCORING_TEMPERATURE = DEFAULT_SCORING_TEMPERATURE
FX_TO_MODEL_CURRENCY = DEFAULT_FX_TO_MODEL_CURRENCY
SCORING_MIN_DENOMINATOR = DEFAULT_SCORING_MIN_DENOMINATOR

NEAR_DEFAULT_CLUSTER = infer_near_default_cluster_from_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

segment_artifact = get_segment_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_PATH:", MODEL_PATH)
print("Scoring segment:", SCORING_SEGMENT)
print("Artifact version:", artifact.get("artifact_version"))
print("Features used by model:", segment_artifact.get("feature_cols"))
print("Cluster labels:", segment_artifact.get("cluster_labels"))
print("Risk rank:", segment_artifact.get("risk_rank"))
print("Inferred near-default cluster:", NEAR_DEFAULT_CLUSTER)

PROJECT_ROOT: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3
MODEL_PATH: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\saved_models\nonfinancial_credit_scorecard_kmeans_k5_v3.joblib
Scoring segment: Non-financial
Artifact version: v3_scorecard
Features used by model: ['earnings_risk', 'structural_distress_risk', 'operating_cashflow_risk', 'liquidity_risk', 'debt_service_risk', 'leverage_risk']
Cluster labels: {1: '1 - Strong relative credit profile', 3: '2 - Good credit profile', 0: '3 - Leveraged / elevated risk profile', 4: '4 - Weak credit profile', 2: '5 - Distressed / near-default proxy'}
Risk rank: {1: 1, 3: 2, 0: 3, 4: 4, 2: 5}
Inferred near-default cluster: 2


## 2. Manual input example

Edit the numbers below to score a private company, competitor, or simulated case.

Currency note: if your model was trained on USD and the input is in EUR, set `FX_TO_MODEL_CURRENCY` to the EUR/USD conversion rate. Ratios are mostly unaffected; `log_assets` is affected.

In [3]:
manual_input_2025 = pd.DataFrame([{
    "company_name": "Manual 2025 Company",
    "fiscal_year": 2025,
    "currency": "USD",
    "major_sector": "Manufacturing / Industrials",
    "financial_flag": "Non-financial",

    # Values converted from thousand BGN to full USD units at 1.7 BGN/USD
    "revenue": 48_585_294.12,
    "assets": 29_721_275.23,
    "current_assets": 10_037_745.82,
    "cash": 34_804.65,
    "receivables": 2_208_235.29,
    "inventory": 7_794_705.88,
    "equity": 9_082_352.94,
    "current_liabilities": 12_722_352.94,
    "liabilities": 20_638_823.53,
    "long_term_debt": 7_910_588.24,
    "short_term_debt": 1_478_235.29,
    "net_income": 949_411.76,
    "cfo": 3_558_235.29,
    "capex": 588_235.29,
    "operating_income": 1_664_705.88,
    "interest_expense": 597_647.06,

    # Optional EBITDA inputs used by the patched v3 scorer.
    # If direct EBITDA is not available, scoring.py can calculate it from
    # operating_income + depreciation_amortization when both are supplied.
    "depreciation_amortization": 2_713_000,
    "ebitda": np.nan,
}])

template_path = INPUT_PATH / "model_2025_company.csv"

manual_input_2025.to_csv(template_path, index=False)
print("Template saved to:", template_path)


Template saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\inputs\model_2025_company.csv


In [4]:
eeec_input_2025 = pd.DataFrame([{
    "company_name": "Eastern European Electric Company B.V.",
    "fiscal_year": 2025,
    "currency": "BGN",
    "major_sector": "Transportation / Utilities",
    "financial_flag": "Non-financial",
    "revenue": 2_572_207_000,
    "assets": 2_127_723_000,
    "current_assets": 1_024_267_000,
    "cash": 224_558_000,
    "receivables": 465_950_000,
    "inventory": 42_958_000,
    "equity": 457_686_000,
    "current_liabilities": 541_002_000,
    "liabilities": 1_670_037_000,
    "long_term_debt": 988_887_000,
    "short_term_debt": 14_753_000,
    "net_income": 36_799_000,
    "cfo": 265_267_000,
    "capex": 212_690_000,
    "operating_income": 54_961_000,
    "interest_expense": 102_302_000,

    # Optional EBITDA inputs used by the patched v3 scorer.
    # If direct EBITDA is not available, scoring.py can calculate it from
    # operating_income + depreciation_amortization when both are supplied.
    "depreciation_amortization": 119_519_000,
    "ebitda": 259_574_000,
}])

template_path = INPUT_PATH / "EEEC_2025.csv"

eeec_input_2025.to_csv(template_path, index=False)
print("Template saved to:", template_path)

Template saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\inputs\EEEC_2025.csv


In [5]:
#Configure input data model
template_path = INPUT_PATH / "EEEC_2025.csv"
current_input = pd.read_csv(template_path)

#Configure output files common name
current_file_name = "EEEC_2025"

display(current_input)

,company_name,fiscal_year,currency,major_sector,financial_flag,revenue,assets,current_assets,cash,receivables,inventory,equity,current_liabilities,liabilities,long_term_debt,short_term_debt,net_income,cfo,capex,operating_income,interest_expense,depreciation_amortization,ebitda
0,Eastern European Electric Company B.V.,2025,BGN,Transportation / Utilities,Non-financial,2572207000,2127723000,1024267000,224558000,465950000,42958000,457686000,541002000,1670037000,988887000,14753000,36799000,265267000,212690000,54961000,102302000,119519000,259574000


In [6]:
scored_manual = score_companies(
    input_df=current_input,
    artifact=artifact,
    segment=SCORING_SEGMENT,
    temperature=SCORING_TEMPERATURE,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
    min_denominator=DEFAULT_SCORING_MIN_DENOMINATOR,
    near_default_cluster=NEAR_DEFAULT_CLUSTER,
)

scored_manual_with_outlook = add_adjacent_bucket_distances_and_outlook(
    scored_manual,
    artifact,
    neutral_band=0.15,
    upgrade_boundary_multiplier=1.35,
    downgrade_boundary_multiplier=1.35,
)

current_external_rating = None # optional, none is not existant
scored_manual_with_outlook = apply_credit_guardrails(
    scored_manual_with_outlook,
    external_rating=current_external_rating
)

existing_summary_cols_with_outlook = [
    col for col in SUMMARY_COLS_WITH_OUTLOOK
    if col in scored_manual_with_outlook.columns
]

missing_summary_cols_with_outlook = [
    col for col in SUMMARY_COLS_WITH_OUTLOOK
    if col not in scored_manual_with_outlook.columns
]

if missing_summary_cols_with_outlook:
    print("Missing summary columns:", missing_summary_cols_with_outlook)

display(scored_manual_with_outlook[existing_summary_cols_with_outlook])

,company_name,fiscal_year,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,scorecard_credit_score,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion
0,Eastern European Electric Company B.V.,2025,0,3 - Leveraged / elevated risk profile,3,0.271027,0.186913,0.370475,3,2 - Good credit profile,0.636156,4,4 - Weak credit profile,0.800406,Neutral,Company remains materially closer to its assig...,59.843817,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ..."


In [7]:
# ---------------------------------------------------------------------
# 8. Company ratio and diagnostic snapshot
# ---------------------------------------------------------------------

ratio_cols = artifact.get("ratio_cols", RATIO_COLS)

existing_ratio_cols = [
    col for col in ratio_cols
    if col in scored_manual.columns
]

missing_ratio_cols = [
    col for col in ratio_cols
    if col not in scored_manual.columns
]

if missing_ratio_cols:
    print("Missing ratio/diagnostic columns:", missing_ratio_cols)

company_ratio_snapshot = scored_manual[
    ["company_name"] + existing_ratio_cols
].copy()

display(company_ratio_snapshot.T.round(4))

,0
company_name,Eastern European Electric Company B.V.
log_assets,21.478318
liabilities_to_assets,0.784894
debt_to_assets,0.471697
debt_to_equity,2.192857
equity_to_assets,0.215106
cash_to_assets,0.105539
net_income_to_assets,0.017295
cfo_to_assets,0.124672
revenue_to_assets,1.208901


## 3. Compare the company to cluster medians

This helps explain why the company was assigned to a specific profile.

In [8]:
comparison_to_cluster = compare_to_cluster_profile(
    scored_manual.iloc[0],
    artifact,
)

comparison_to_cluster

,company_value,assigned_cluster_median,difference,relative_position
metric,,,,
log_assets,21.4783,22.5017,-1.0234,below_cluster_median
liabilities_to_assets,0.7849,0.6993,0.0856,above_cluster_median
debt_to_assets,0.4717,0.3701,0.1016,above_cluster_median
equity_to_assets,0.2151,0.2781,-0.0630,below_cluster_median
current_ratio,1.8933,1.1178,0.7754,above_cluster_median
quick_ratio,1.2764,0.4644,0.8120,above_cluster_median
cash_to_assets,0.1055,0.0361,0.0694,above_cluster_median
net_income_to_assets,0.0173,0.0248,-0.0075,below_cluster_median
cfo_to_assets,0.1247,0.0649,0.0597,above_cluster_median


## 4. Scenario analysis

This section shows how a company migrates across clusters under stress cases

In [9]:
# ---------------------------------------------------------------------
# 10. Build scenario inputs
# ---------------------------------------------------------------------

scenario_input = make_scenarios(current_input.iloc[0])

scenario_input_cols = [
    "scenario",
    "assets",
    "liabilities",
    "equity",
    "cash",
    "revenue",
    "net_income",
    "cfo",
    "long_term_debt",
    "short_term_debt",
    "operating_income",
    "interest_expense",
    "depreciation_amortization",
    "ebitda",
]

existing_scenario_input_cols = [
    col for col in scenario_input_cols
    if col in scenario_input.columns
]

missing_scenario_input_cols = [
    col for col in scenario_input_cols
    if col not in scenario_input.columns
]

if missing_scenario_input_cols:
    print("Missing scenario input columns:", missing_scenario_input_cols)

display(scenario_input[existing_scenario_input_cols])

,scenario,assets,liabilities,equity,cash,revenue,net_income,cfo,long_term_debt,short_term_debt,operating_income,interest_expense,depreciation_amortization,ebitda
0,base,2127723000,1.670037e+09,457686000.0,224558000.0,2.572207e+09,36799000.0,265267000.0,9.888870e+08,14753000.0,54961000.0,102302000.0,119519000,259574000
0,revenue_down_15pct,2127723000,1.670037e+09,457686000.0,224558000.0,2.186376e+09,25759300.0,198950250.0,9.888870e+08,14753000.0,38472700.0,102302000.0,119519000,259574000
0,debt_up_25pct,2127723000,1.917259e+09,457686000.0,175113650.0,2.572207e+09,36799000.0,265267000.0,1.236109e+09,14753000.0,54961000.0,102302000.0,119519000,259574000
0,cash_burn_case,2127723000,1.670037e+09,457686000.0,112279000.0,2.572207e+09,-36799000.0,-265267000.0,9.888870e+08,14753000.0,54961000.0,102302000.0,119519000,259574000
0,near_default_stress,2127723000,2.340495e+09,-212772300.0,224558000.0,2.572207e+09,36799000.0,265267000.0,1.595792e+09,319158450.0,40920800.0,102302000.0,119519000,259574000


In [10]:
# ---------------------------------------------------------------------
# 11. Score scenarios
# ---------------------------------------------------------------------

scored_scenarios = score_companies(
    input_df=scenario_input,
    artifact=artifact,
    segment=SCORING_SEGMENT,
    temperature=SCORING_TEMPERATURE,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
    min_denominator=SCORING_MIN_DENOMINATOR,
    near_default_cluster=NEAR_DEFAULT_CLUSTER,
)

scored_scenarios = add_adjacent_bucket_distances_and_outlook(
    scored_scenarios,
    artifact,
    neutral_band=0.15,
    upgrade_boundary_multiplier=1.35,
    downgrade_boundary_multiplier=1.35,
)

scored_scenarios = apply_credit_guardrails(
    scored_scenarios,
    external_rating=current_external_rating,
)

scenario_summary_cols = artifact.get("scenario_summary_cols", SCENARIO_SUMMARY_COLS)

# Add selected diagnostic ratios for direct scenario comparison.
scenario_diagnostic_cols = [
    "liabilities_to_assets",
    "debt_to_assets",
    "debt_to_equity",
    "equity_to_assets",
    "cfo_to_assets",
    "current_ratio",
    "quick_ratio",
    "interest_coverage",
    "fcf_to_debt",
    "ebitda",
    "ebitda_margin",
    "debt_to_ebitda",
    "net_debt_to_ebitda",
    "ebitda_interest_coverage",
    "debt_service_risk",
]

scenario_display_cols = []

for col in list(scenario_summary_cols) + scenario_diagnostic_cols:
    if col in scored_scenarios.columns and col not in scenario_display_cols:
        scenario_display_cols.append(col)

missing_scenario_cols = [
    col for col in list(scenario_summary_cols) + scenario_diagnostic_cols
    if col not in scored_scenarios.columns
]

if missing_scenario_cols:
    print("Missing scenario columns:", missing_scenario_cols)

display(scored_scenarios[scenario_display_cols])

,scenario,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,scorecard_credit_score,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion,liabilities_to_assets,debt_to_assets,debt_to_equity,equity_to_assets,cfo_to_assets,current_ratio,quick_ratio,interest_coverage,fcf_to_debt,ebitda,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage,debt_service_risk
0,base,0,3 - Leveraged / elevated risk profile,3,0.271027,0.186913,0.370475,3,2 - Good credit profile,0.636156,4.0,4 - Weak credit profile,0.800406,Neutral,Company remains materially closer to its assig...,59.843817,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",0.784894,0.471697,2.192857,0.215106,0.124672,1.893278,1.276350,0.537243,0.052386,259574000.0,0.100915,3.866489,3.001387,2.537331,0.804994
1,revenue_down_15pct,0,3 - Leveraged / elevated risk profile,3,0.272840,0.203360,0.340994,3,2 - Good credit profile,0.688381,4.0,4 - Weak credit profile,0.716764,Neutral,Company remains materially closer to its assig...,63.849901,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",0.784894,0.471697,2.192857,0.215106,0.093504,1.893278,1.276350,0.376070,-0.013690,259574000.0,0.118723,3.866489,3.001387,2.537331,0.838033
2,debt_up_25pct,0,3 - Leveraged / elevated risk profile,3,0.276613,0.205891,0.442690,3,2 - Good credit profile,0.817395,4.0,4 - Weak credit profile,0.883393,Neutral,Company remains materially closer to its assig...,67.168450,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",0.901085,0.587887,2.733013,0.215106,0.124672,1.893278,1.184956,0.537243,0.042033,259574000.0,0.100915,4.818902,4.144283,2.537331,0.857792
3,cash_burn_case,2,5 - Distressed / near-default proxy,5,0.289240,0.289240,0.347174,4,4 - Weak credit profile,0.577565,NaN,None,NaN,Neutral,Only the stronger adjacent bucket is available...,78.678130,1.0,"interest_coverage_below_1, negative_cfo_to_assets",Override required,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to the weakest model-r...,The cluster assignment appears primarily suppo...,Manual analyst override is required. The model...,0.784894,0.471697,2.192857,0.215106,-0.124672,1.893278,1.068811,0.537243,-0.476224,259574000.0,0.100915,3.866489,3.433938,2.537331,0.881188
4,near_default_stress,0,3 - Leveraged / elevated risk profile,3,0.265110,0.231848,0.721894,3,2 - Good credit profile,1.124772,4.0,4 - Weak credit profile,1.109415,Neutral,Company remains materially closer to its assig...,75.954216,1.0,"liabilities_exceed_assets, negative_equity, hi...",Override required,"elevated_debt_to_assets, high_debt_to_assets, ...",Structural balance-sheet distress indicators w...,The cluster assignment appears primarily suppo...,Manual analyst override is required. The model...,1.100000,0.900000,-9.000000,-0.100000,0.124672,1.893278,1.276350,0.400000,0.027456,259574000.0,0.100915,7.377282,6.512180,2.537331,0.924135


## 5. Save report outputs to files


In [11]:
# ---------------------------------------------------------------------
# 12. Build and save credit report outputs
# ---------------------------------------------------------------------

report_tables = build_credit_report_tables(
    scored_manual_with_outlook=scored_manual_with_outlook,
    scored_manual=scored_manual,
    comparison_to_cluster=comparison_to_cluster,
    scenario_input=scenario_input,
    scored_scenarios=scored_scenarios,
    artifact=artifact,
)

saved_paths = save_credit_report_outputs(
    tables=report_tables,
    output_path=OUTPUT_PATH,
    base_filename=current_file_name,
)

print("Scores saved to:", saved_paths["score_csv"])
print("Scenario CSV saved to:", saved_paths["scenario_csv"])
print("Full Excel score report saved to:", saved_paths["report_xlsx"])

display(report_tables["scored_file"])
display(report_tables["scenario_file"])

Scores saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\EEEC_2025_score_result.csv
Scenario CSV saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\EEEC_2025_scenario_analysis.csv
Full Excel score report saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\EEEC_2025_score_report.xlsx


,company_name,fiscal_year,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion
0,Eastern European Electric Company B.V.,2025,0,3 - Leveraged / elevated risk profile,3,0.271027,0.186913,0.370475,3,2 - Good credit profile,0.636156,4,4 - Weak credit profile,0.800406,Neutral,Company remains materially closer to its assig...,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ..."


,scenario,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,scorecard_credit_score,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion,log_assets,liabilities_to_assets,debt_to_assets,debt_to_equity,equity_to_assets,cash_to_assets,net_income_to_assets,cfo_to_assets,revenue_to_assets,current_ratio,quick_ratio,interest_coverage,fcf_to_debt,operating_margin,gross_margin,cfo_to_liabilities,capex_to_revenue,total_debt,net_debt,ebitda,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage,leverage_risk,liquidity_risk,earnings_risk,operating_cashflow_risk,debt_service_risk,debt_service_risk_legacy,structural_distress_risk
0,base,0,3 - Leveraged / elevated risk profile,3,0.271027,0.186913,0.370475,3,2 - Good credit profile,0.636156,4.0,4 - Weak credit profile,0.800406,Neutral,Company remains materially closer to its assig...,59.843817,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",21.478318,0.784894,0.471697,2.192857,0.215106,0.105539,0.017295,0.124672,1.208901,1.893278,1.276350,0.537243,0.052386,0.021367,NaN,0.158839,0.082688,1.003640e+09,7.790820e+08,259574000.0,0.100915,3.866489,3.001387,2.537331,0.625658,0.350685,0.551367,0.440049,0.804994,0.878091,0.699837
1,revenue_down_15pct,0,3 - Leveraged / elevated risk profile,3,0.272840,0.203360,0.340994,3,2 - Good credit profile,0.688381,4.0,4 - Weak credit profile,0.716764,Neutral,Company remains materially closer to its assig...,63.849901,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",21.478318,0.784894,0.471697,2.192857,0.215106,0.105539,0.012107,0.093504,1.027566,1.893278,1.276350,0.376070,-0.013690,0.017597,NaN,0.119129,0.097280,1.003640e+09,7.790820e+08,259574000.0,0.118723,3.866489,3.001387,2.537331,0.625658,0.378217,0.585957,0.559347,0.838033,0.930952,0.699837
2,debt_up_25pct,0,3 - Leveraged / elevated risk profile,3,0.276613,0.205891,0.442690,3,2 - Good credit profile,0.817395,4.0,4 - Weak credit profile,0.883393,Neutral,Company remains materially closer to its assig...,67.168450,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",21.478318,0.901085,0.587887,2.733013,0.215106,0.082301,0.017295,0.124672,1.208901,1.893278,1.184956,0.537243,0.042033,0.021367,NaN,0.138357,0.082688,1.250862e+09,1.075748e+09,259574000.0,0.100915,4.818902,4.144283,2.537331,0.795404,0.400973,0.551367,0.485081,0.857792,0.886374,0.771339
3,cash_burn_case,2,5 - Distressed / near-default proxy,5,0.289240,0.289240,0.347174,4,4 - Weak credit profile,0.577565,NaN,None,NaN,Neutral,Only the stronger adjacent bucket is available...,78.678130,1.0,"interest_coverage_below_1, negative_cfo_to_assets",Override required,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to the weakest model-r...,The cluster assignment appears primarily suppo...,Manual analyst override is required. The model...,21.478318,0.784894,0.471697,2.192857,0.215106,0.052770,-0.017295,-0.124672,1.208901,1.893278,1.068811,0.537243,-0.476224,0.021367,NaN,-0.158839,0.082688,1.003640e+09,8.913610e+08,259574000.0,0.100915,3.866489,3.433938,2.537331,0.652692,0.510245,0.781967,1.000000,0.881188,1.000000,0.699837
4,near_default_stress,0,3 - 

In [12]:
guardrail_cols = [
    "guardrail_level",
    "guardrail_flags",
    "guardrail_summary",
    "analyst_interpretation",
    "commercial_conclusion",
]

display(scored_manual_with_outlook[guardrail_cols])

,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion
0,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ..."


In [13]:
pdf_path = save_credit_pdf_report(
    report_tables=report_tables,
    scored_manual_with_outlook=scored_manual_with_outlook,
    comparison_to_cluster=comparison_to_cluster,
    scored_scenarios=scored_scenarios,
    artifact=artifact,
    output_path=OUTPUT_PATH,
    base_filename=current_file_name,
)

print("PDF credit report saved to:", pdf_path)

PDF credit report saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\EEEC_2025_credit_report.pdf


## 6. Optional: inspect saved artifact internals

In [14]:
# ---------------------------------------------------------------------
# Optional artifact inspection
# ---------------------------------------------------------------------

nonfin_artifact = get_segment_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

print("Artifact version:")
print(artifact.get("artifact_version"))

print("\nPrimary segment:")
print(artifact.get("primary_segment"))

print("\nFeatures used by model:")
print(nonfin_artifact.get("feature_cols"))

print("\nCluster labels:")
print(nonfin_artifact.get("cluster_labels"))

print("\nRisk rank:")
print(nonfin_artifact.get("risk_rank"))

print("\nCluster sizes:")
display(nonfin_artifact.get("cluster_sizes"))

print("\nCluster profile:")
display(nonfin_artifact.get("cluster_profile"))

print("\nCluster profile ranked:")
display(nonfin_artifact.get("cluster_profile_ranked"))

Artifact version:
v3_scorecard

Primary segment:
Non-financial

Features used by model:
['earnings_risk', 'structural_distress_risk', 'operating_cashflow_risk', 'liquidity_risk', 'debt_service_risk', 'leverage_risk']

Cluster labels:
{1: '1 - Strong relative credit profile', 3: '2 - Good credit profile', 0: '3 - Leveraged / elevated risk profile', 4: '4 - Weak credit profile', 2: '5 - Distressed / near-default proxy'}

Risk rank:
{1: 1, 3: 2, 0: 3, 4: 4, 2: 5}

Cluster sizes:


,cluster,issuer_years,issuers
0,0,3419,1204
1,1,1640,639
2,2,2357,1088
3,3,2052,874
4,4,2706,1114



Cluster profile:


,scorecard_credit_score,structural_distress_risk,earnings_risk,operating_cashflow_risk,liquidity_risk,leverage_risk,debt_service_risk,liabilities_risk,debt_load_risk,equity_buffer_risk,cash_buffer_risk,current_liquidity_risk,quick_liquidity_risk,profitability_risk,cashflow_risk,cfo_to_debt_risk,coverage_risk,fcf_risk,debt_repayment_risk,ebitda_margin_risk,debt_to_ebitda_risk,net_debt_to_ebitda_risk,ebitda_coverage_risk,negative_ebitda_flag,negative_equity_flag,liabilities_exceed_assets_flag,log_assets,small_company_flag,mid_company_flag,large_company_flag,total_debt,net_debt,ebitda,liabilities_to_assets,debt_to_assets,debt_to_equity,equity_to_assets,cash_to_assets,net_income_to_assets,cfo_to_assets,revenue_to_assets,current_ratio,quick_ratio,interest_coverage,fcf_to_debt,operating_margin,gross_margin,cfo_to_liabilities,capex_to_revenue,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage,debt_service_risk_legacy
cluster,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,60.246444,0.595494,0.501552,0.609670,0.671487,0.576033,0.759662,0.614266,0.415515,0.572180,0.785197,0.691077,0.828501,0.501552,0.540251,0.719896,0.860842,0.54391,0.566573,0.418371,0.804575,0.822035,0.847918,0.0,0.0,0.0,22.501697,0.0,0.0,1.0,1.840000e+09,1.411280e+09,399857000.0,0.699273,0.370085,1.080044,0.278083,0.036146,0.024767,0.064937,0.414955,1.117846,0.464374,2.358570,0.128045,0.118968,0.315761,0.095284,0.025416,0.182652,5.022875,4.288140,3.889554,0.726338
1,21.236755,0.331364,0.154353,0.207244,0.377506,0.218323,0.108311,0.321549,0.079934,0.336115,0.430949,0.358071,0.513642,0.154353,0.323700,0.000000,0.000000,0.00000,0.000000,0.431171,0.036720,0.000000,0.113917,0.0,0.0,0.0,21.850772,0.0,0.0,1.0,4.444425e+08,7.403450e+07,447245000.0,0.509007,0.151957,0.317343,0.431525,0.087512,0.076847,0.119075,0.741405,1.783857,0.857948,14.273972,0.596364,0.142820,0.366556,0.237438,0.023953,0.177532,1.183598,0.370943,17.835574,0.073317
2,90.199546,0.877532,1.000000,1.000000,0.758301,0.785994,1.000000,0.913472,0.408880,0.926550,0.480905,0.752428,0.870030,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,0.0,0.0,19.158992,0.0,0.0,1.0,5.940900e+07,2.622005e+07,-9420000.0,0.893757,0.365772,0.272087,0.047742,0.080269,-0.172683,-0.068450,0.517020,0.995143,0.412462,-3.598703,-0.328566,-0.244279,0.277608,-0.085840,0.012766,-0.189852,11.216977,9.613792,-2.711361,1.000000
3,32.030286,0.235268,0.381657,0.291315,0.263647,0.193152,0.577043,0.241922,0.098969,0.236518,0.335939,0.152598,0.256260,0.381657,0.421689,0.078795,0.831867,0.00000,0.000000,0.527238,0.205790,0.000000,0.776479,0.0,0.0,0.0,21.452476,0.0,0.0,1.0,3.014357e+08,6.376900e+07,138675000.0,0.457249,0.164330,0.327429,0.496263,0.101289,0.042751,0.094578,0.601971,2.194803,1.179676,2.683085,0.410472,0.095797,0.362174,0.208390,0.022021,0.139105,2.028950,0.710059,5.246892,0.597362
4,78.571462,0.225139,1.000000,1.000000,0.260315,0.423593,1.000000,0.176472,0.061918,0.209834,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,0.0,0.0,19.361843,0.0,0.0,1.0,3.023725e+07,-3.132750e+06,-21648750.0,0.414707,0.140247,0.252518,0.513608,0.178916,-0.154447,-0.085228,0.324524,2.979425,1.516596,-14.226752,-0.987147,-0.333985,0.352504,-0.248687,0.025842,-0.268102,8.356646,5.322432,-12.033248,1.000000



Cluster profile ranked:


,financial_flag,cluster,issuer_years,issuers,median_log_assets,median_liabilities_to_assets,median_debt_to_assets,median_debt_to_equity,median_equity_to_assets,median_cash_to_assets,median_net_income_to_assets,median_cfo_to_assets,median_revenue_to_assets,median_current_ratio,median_quick_ratio,median_interest_coverage,median_fcf_to_debt,median_operating_margin,median_gross_margin,median_ebitda_margin,median_debt_to_ebitda,median_net_debt_to_ebitda,median_ebitda_interest_coverage,median_leverage_risk,median_liquidity_risk,median_earnings_risk,median_operating_cashflow_risk,median_debt_service_risk,median_structural_distress_risk,median_scorecard_credit_score,rating_style_rank,rating_style_label
1,Non-financial,1,1640,639,21.850772,0.509007,0.151957,0.317343,0.431525,0.087512,0.076847,0.119075,0.741405,1.783857,0.857948,14.273972,0.596364,0.142820,0.366556,0.177532,1.183598,0.370943,17.835574,0.218323,0.377506,0.154353,0.207244,0.108311,0.331364,21.236755,1,1 - Strong relative credit profile
3,Non-financial,3,2052,874,21.452476,0.457249,0.164330,0.327429,0.496263,0.101289,0.042751,0.094578,0.601971,2.194803,1.179676,2.683085,0.410472,0.095797,0.362174,0.139105,2.028950,0.710059,5.246892,0.193152,0.263647,0.381657,0.291315,0.577043,0.235268,32.030286,2,2 - Good credit profile
0,Non-financial,0,3419,1204,22.501697,0.699273,0.370085,1.080044,0.278083,0.036146,0.024767,0.064937,0.414955,1.117846,0.464374,2.358570,0.128045,0.118968,0.315761,0.182652,5.022875,4.288140,3.889554,0.576033,0.671487,0.501552,0.609670,0.759662,0.595494,60.246444,3,3 - Leveraged / elevated risk profile
4,Non-financial,4,2706,1114,19.361843,0.414707,0.140247,0.252518,0.513608,0.178916,-0.154447,-0.085228,0.324524,2.979425,1.516596,-14.226752,-0.987147,-0.333985,0.352504,-0.268102,8.356646,5.322432,-12.033248,0.423593,0.260315,1.000000,1.000000,1.000000,0.225139,78.571462,4,4 - Weak credit profile
2,Non-financial,2,2357,1088,19.158992,0.893757,0.365772,0.272087,0.047742,0.080269,-0.172683,-0.068450,0.517020,0.995143,0.412462,-3.598703,-0.328566,-0.244279,0.277608,-0.189852,11.216977,9.613792,-2.711361,0.785994,0.758301,1.000000,1.000000,1.000000,0.877532,90.199546,5,5 - Distressed / near-default proxy
